# Optuna Hyperparameter Search — Session XLNet

Automated search over key hyperparameters using Optuna (TPE sampler + MedianPruner).
Forked from `model_training_BCE.ipynb`.

In [ ]:
import os
import gc
import json
import datetime

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LambdaLR

from transformers import XLNetConfig, XLNetModel
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

import optuna
from optuna.exceptions import TrialPruned

device = torch.device("cuda")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
print(f"Optuna version: {optuna.__version__}")

## Fixed Configuration

In [ ]:
# ---------------------------------------------------------------------------
# Feature configuration — these are FIXED (not tuned)
# ---------------------------------------------------------------------------
CATEGORICAL_FEATURES = {
    "sess_pid_seq":  {"cardinality": 164386, "embedding_dim": 64},
    "sess_csid_seq": {"cardinality": 622,    "embedding_dim": 32},
    "sess_ccid_seq": {"cardinality": 129,    "embedding_dim": 16},
    "sess_bid_seq":  {"cardinality": 3426,   "embedding_dim": 32},
}

CONTINUOUS_FEATURES = [
    "sess_price_log_norm_seq",
    "sess_dtime_secs_log_norm_seq",
    "sess_prod_recency_days_log_norm_seq",
]

STATIC_CATEGORICAL_FEATURES = {
    "user_segment": {"cardinality": 10, "embedding_dim": 8},
    "device_type":  {"cardinality": 5,  "embedding_dim": 4},
}

STATIC_CONTINUOUS_FEATURES = [
    "user_lifetime_days",
    "user_avg_order_value",
]

TARGET_COLUMN = "target"

HAS_STATIC = bool(STATIC_CATEGORICAL_FEATURES) or bool(STATIC_CONTINUOUS_FEATURES)

# ---------------------------------------------------------------------------
# Search configuration
# ---------------------------------------------------------------------------
N_TRIALS = 200
MAX_SEQ_LEN = 20       # fixed
EVAL_BATCH_SIZE = 512   # fixed (only affects speed, not quality)

# MLM target features — fixed (which features to mask)
MLM_TARGET_FEATURES = [
    "sess_pid_seq",
    "sess_csid_seq",
]

## Load Data

In [ ]:
# ---------------------------------------------------------------------------
# Load your data here.
# ---------------------------------------------------------------------------

# train_df = pd.read_parquet("train.parquet")
# test_df = pd.read_parquet("test.parquet")

print(f"Train: {len(train_df)} sessions")
print(f"Test:  {len(test_df)} sessions")

## Dataset

In [ ]:
class SessionDataset(Dataset):
    """Wraps a pandas DataFrame of sessions into a PyTorch Dataset."""

    def __init__(self, df, max_seq_len=MAX_SEQ_LEN):
        self.df = df.reset_index(drop=True)
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq_len = min(int(row["sess_seq_len"]), self.max_seq_len)
        result = {}

        for feat_name in CATEGORICAL_FEATURES:
            arr = np.array(row[feat_name], dtype=np.int64)[-self.max_seq_len:]
            padded = np.zeros(self.max_seq_len, dtype=np.int64)
            padded[: len(arr)] = arr
            result[feat_name] = torch.tensor(padded, dtype=torch.long)

        cont_arrays = []
        for feat_name in CONTINUOUS_FEATURES:
            arr = np.array(row[feat_name], dtype=np.float32)[-self.max_seq_len:]
            padded = np.zeros(self.max_seq_len, dtype=np.float32)
            padded[: len(arr)] = arr
            cont_arrays.append(padded)
        result["continuous"] = torch.tensor(
            np.stack(cont_arrays, axis=-1), dtype=torch.float32
        )

        mask = np.zeros(self.max_seq_len, dtype=np.float32)
        mask[:seq_len] = 1.0
        result["attention_mask"] = torch.tensor(mask, dtype=torch.float32)

        for feat_name in STATIC_CATEGORICAL_FEATURES:
            result[f"static_{feat_name}"] = torch.tensor(
                int(row[feat_name]), dtype=torch.long
            )

        if STATIC_CONTINUOUS_FEATURES:
            static_cont = np.array(
                [float(row[f]) for f in STATIC_CONTINUOUS_FEATURES], dtype=np.float32
            )
            result["static_continuous"] = torch.tensor(static_cont, dtype=torch.float32)

        result["target"] = torch.tensor(
            float(row[TARGET_COLUMN]), dtype=torch.float32
        )

        return result


# Pre-build datasets (shared across all trials)
train_ds = SessionDataset(train_df)
test_ds = SessionDataset(test_df)
print(f"Train dataset: {len(train_ds)} sessions")
print(f"Test  dataset: {len(test_ds)} sessions")

## Parameterized Model

In [ ]:
class SessionXLNetModel(nn.Module):
    """XLNet-based session model — accepts hyperparameters as constructor args."""

    def __init__(self, hp):
        super().__init__()
        self.hp = hp

        # === Sequential feature embeddings ===
        self.embeddings = nn.ModuleDict()
        for feat, cfg in CATEGORICAL_FEATURES.items():
            self.embeddings[feat] = nn.Embedding(
                cfg["cardinality"], cfg["embedding_dim"], padding_idx=0
            )

        self.continuous_proj = nn.Sequential(
            nn.Linear(len(CONTINUOUS_FEATURES), hp["continuous_proj_dim"]),
            nn.ReLU(),
        )

        cat_dim = sum(c["embedding_dim"] for c in CATEGORICAL_FEATURES.values())
        total_input_dim = cat_dim + hp["continuous_proj_dim"]
        self.input_mlp = nn.Sequential(
            nn.Linear(total_input_dim, hp["d_model"]),
            nn.ReLU(),
        )

        # === XLNet backbone ===
        xlnet_config = XLNetConfig(
            vocab_size=2,
            d_model=hp["d_model"],
            n_layer=hp["n_layer"],
            n_head=hp["n_head"],
            d_inner=4 * hp["d_model"],
            ff_activation="gelu",
            attn_type="bi",
            mem_len=0,
            dropout=hp["dropout"],
        )
        self.xlnet = XLNetModel(xlnet_config)

        # === Static feature branch ===
        classifier_input_dim = hp["d_model"]

        if HAS_STATIC:
            self.static_embeddings = nn.ModuleDict()
            for feat, cfg in STATIC_CATEGORICAL_FEATURES.items():
                self.static_embeddings[feat] = nn.Embedding(
                    cfg["cardinality"], cfg["embedding_dim"], padding_idx=0
                )

            static_cat_dim = sum(
                c["embedding_dim"] for c in STATIC_CATEGORICAL_FEATURES.values()
            )
            static_cont_dim = len(STATIC_CONTINUOUS_FEATURES)
            static_raw_dim = static_cat_dim + static_cont_dim

            self.static_mlp = nn.Sequential(
                nn.Linear(static_raw_dim, hp["static_proj_dim"]),
                nn.ReLU(),
            )

            self.seq_norm = nn.LayerNorm(hp["d_model"])
            self.static_norm = nn.LayerNorm(hp["static_proj_dim"])

            classifier_input_dim = hp["d_model"] + hp["static_proj_dim"]

        # === MLM auxiliary loss ===
        self.mlm_heads = nn.ModuleDict()
        if hp["mask_prob"] > 0 and MLM_TARGET_FEATURES:
            self.mask_embedding = nn.Parameter(torch.randn(hp["d_model"]))
            for feat in MLM_TARGET_FEATURES:
                card = CATEGORICAL_FEATURES[feat]["cardinality"]
                self.mlm_heads[feat] = nn.Linear(hp["d_model"], card)
            self.mlm_loss_fn = nn.CrossEntropyLoss()

        # === Classification head ===
        self.classifier = nn.Sequential(
            nn.Linear(classifier_input_dim, hp["classifier_hidden_dim"]),
            nn.ReLU(),
            nn.Linear(hp["classifier_hidden_dim"], 1),
        )

        self.loss_fn = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([hp["pos_weight"]])
        )

    def forward(self, batch):
        hp = self.hp
        cat_embeds = [self.embeddings[f](batch[f]) for f in CATEGORICAL_FEATURES]
        cont_proj = self.continuous_proj(batch["continuous"])

        x = torch.cat(cat_embeds + [cont_proj], dim=-1)
        x = self.input_mlp(x)

        attention_mask = batch["attention_mask"]

        # --- MLM masking ---
        mlm_loss = None
        if self.training and hp["mask_prob"] > 0 and self.mlm_heads:
            mask_probs = hp["mask_prob"] * attention_mask
            mlm_mask = torch.bernoulli(mask_probs).bool()

            if mlm_mask.any():
                mlm_targets = {}
                for feat in MLM_TARGET_FEATURES:
                    mlm_targets[feat] = batch[feat][mlm_mask]
                x = x.clone()
                x[mlm_mask] = self.mask_embedding

        # --- XLNet forward ---
        xlnet_out = self.xlnet(inputs_embeds=x, attention_mask=attention_mask)
        hidden = xlnet_out.last_hidden_state

        # --- MLM loss ---
        if self.training and hp["mask_prob"] > 0 and self.mlm_heads:
            if mlm_mask.any():
                masked_hidden = hidden[mlm_mask]
                feat_losses = []
                for feat in MLM_TARGET_FEATURES:
                    mlm_logits = self.mlm_heads[feat](masked_hidden)
                    feat_losses.append(self.mlm_loss_fn(mlm_logits, mlm_targets[feat]))
                mlm_loss = torch.stack(feat_losses).mean()

        # --- Pooling ---
        mask_expanded = attention_mask.unsqueeze(-1)
        hidden_masked = hidden * mask_expanded
        pooled = hidden_masked.sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1)

        # --- Static features ---
        if HAS_STATIC:
            pooled = self.seq_norm(pooled)
            static_parts = []
            for feat in STATIC_CATEGORICAL_FEATURES:
                static_parts.append(self.static_embeddings[feat](batch[f"static_{feat}"]))
            if STATIC_CONTINUOUS_FEATURES:
                static_parts.append(batch["static_continuous"])
            static_cat = torch.cat(static_parts, dim=-1)
            static_repr = self.static_mlp(static_cat)
            static_repr = self.static_norm(static_repr)
            pooled = torch.cat([pooled, static_repr], dim=-1)

        # --- Classifier ---
        logits = self.classifier(pooled).squeeze(-1)
        targets = batch["target"]
        loss = self.loss_fn(logits, targets)

        if mlm_loss is not None:
            loss = loss + hp["mlm_weight"] * mlm_loss

        return loss, logits

## Optuna Objective

In [ ]:
def objective(trial):
    # ------------------------------------------------------------------
    # Sample hyperparameters
    # ------------------------------------------------------------------
    d_model_raw = trial.suggest_int("d_model_mult", 4, 16)  # multiples of 8*n_head
    n_head = trial.suggest_categorical("n_head", [4, 8])
    d_model = d_model_raw * n_head * 8  # ensure divisible by n_head

    hp = {
        "d_model":              d_model,
        "n_head":               n_head,
        "n_layer":              trial.suggest_int("n_layer", 1, 4),
        "continuous_proj_dim":  trial.suggest_categorical("continuous_proj_dim", [32, 64, 128]),
        "classifier_hidden_dim": trial.suggest_categorical("classifier_hidden_dim", [64, 128, 256]),
        "static_proj_dim":      trial.suggest_categorical("static_proj_dim", [32, 64, 128]),
        "dropout":              trial.suggest_float("dropout", 0.0, 0.3, step=0.05),
        "learning_rate":        trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True),
        "weight_decay":         trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "pos_weight":           trial.suggest_float("pos_weight", 5.0, 50.0, step=1.0),
        "mask_prob":            trial.suggest_float("mask_prob", 0.0, 0.3, step=0.05),
        "mlm_weight":           trial.suggest_float("mlm_weight", 0.01, 0.5, log=True),
        "num_epochs":           trial.suggest_int("num_epochs", 5, 20),
        "train_batch_size":     trial.suggest_categorical("train_batch_size", [128, 256, 512]),
    }

    trial.set_user_attr("d_model", d_model)

    # ------------------------------------------------------------------
    # Build model & data loaders
    # ------------------------------------------------------------------
    model = SessionXLNetModel(hp).to(device)

    train_loader = DataLoader(
        train_ds, batch_size=hp["train_batch_size"], shuffle=True, drop_last=False,
    )
    test_loader = DataLoader(
        test_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=hp["learning_rate"], weight_decay=hp["weight_decay"],
    )
    total_steps = len(train_loader) * hp["num_epochs"]
    scheduler = LambdaLR(
        optimizer,
        lr_lambda=lambda step: max(0.0, 1 - step / max(total_steps, 1)),
    )

    # ------------------------------------------------------------------
    # Training loop with per-epoch pruning
    # ------------------------------------------------------------------
    best_auc = 0.0

    for epoch in range(1, hp["num_epochs"] + 1):
        # --- Train ---
        model.train()
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()
            loss, _ = model(batch)
            loss.backward()
            optimizer.step()
            scheduler.step()

        # --- Evaluate ---
        model.eval()
        all_logits, all_targets = [], []
        with torch.no_grad():
            for batch in test_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                _, logits = model(batch)
                all_logits.append(logits.cpu())
                all_targets.append(batch["target"].cpu())

        all_logits = torch.cat(all_logits)
        all_targets = torch.cat(all_targets)
        probs = torch.sigmoid(all_logits).numpy()
        targets_np = all_targets.numpy()

        try:
            auc = roc_auc_score(targets_np, probs)
        except ValueError:
            auc = 0.0

        best_auc = max(best_auc, auc)

        # Report to Optuna for pruning
        trial.report(auc, epoch)
        if trial.should_prune():
            del model, optimizer, scheduler
            gc.collect()
            torch.cuda.empty_cache()
            raise TrialPruned()

    # Cleanup
    del model, optimizer, scheduler
    gc.collect()
    torch.cuda.empty_cache()

    return best_auc

## Run Study

In [ ]:
time_stamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M")
study_name = f"xlnet_hpsearch_{time_stamp}"

study = optuna.create_study(
    study_name=study_name,
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=10,
        n_warmup_steps=3,
    ),
    storage=f"sqlite:///{study_name}.db",
    load_if_exists=True,
)

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nCompleted {len(study.trials)} trials")
print(f"Best AUC: {study.best_value:.6f}")
print(f"Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")
if "d_model" in study.best_trial.user_attrs:
    print(f"  d_model (derived): {study.best_trial.user_attrs['d_model']}")

## Results Analysis

In [ ]:
# Trial history DataFrame
trials_df = study.trials_dataframe()
trials_df = trials_df.sort_values("value", ascending=False)

print(f"Top 10 trials:")
display_cols = ["number", "value", "state"] + [c for c in trials_df.columns if c.startswith("params_")]
print(trials_df[display_cols].head(10).to_string())

# Save full results
results_path = f"optuna_results_{time_stamp}.csv"
trials_df.to_csv(results_path, index=False)
print(f"\nFull results saved to {results_path}")

## Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# --- 1. Optimization history ---
ax = axes[0, 0]
completed = trials_df[trials_df["state"] == "COMPLETE"].copy()
ax.scatter(completed["number"], completed["value"], s=10, alpha=0.6)
# Running best
running_best = completed["value"].cummax()
ax.plot(completed["number"], running_best, color="red", linewidth=2, label="Best so far")
ax.set_xlabel("Trial")
ax.set_ylabel("AUC")
ax.set_title("Optimization History")
ax.legend()
ax.grid(True, alpha=0.3)

# --- 2. Parameter importances ---
ax = axes[0, 1]
try:
    importances = optuna.importance.get_param_importances(study)
    names = list(importances.keys())[:12]
    values = [importances[n] for n in names]
    ax.barh(names[::-1], values[::-1])
    ax.set_xlabel("Importance")
    ax.set_title("Hyperparameter Importance")
except Exception as e:
    ax.text(0.5, 0.5, f"Could not compute\nimportances:\n{e}",
            ha="center", va="center", transform=ax.transAxes)

# --- 3. Learning rate vs AUC ---
ax = axes[1, 0]
if "params_learning_rate" in completed.columns:
    ax.scatter(completed["params_learning_rate"], completed["value"], s=10, alpha=0.5)
    ax.set_xscale("log")
    ax.set_xlabel("Learning Rate")
    ax.set_ylabel("AUC")
    ax.set_title("Learning Rate vs AUC")
    ax.grid(True, alpha=0.3)

# --- 4. pos_weight vs AUC ---
ax = axes[1, 1]
if "params_pos_weight" in completed.columns:
    ax.scatter(completed["params_pos_weight"], completed["value"], s=10, alpha=0.5)
    ax.set_xlabel("pos_weight")
    ax.set_ylabel("AUC")
    ax.set_title("pos_weight vs AUC")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Export Best Hyperparameters

In [ ]:
best = study.best_trial
d_model = best.user_attrs.get("d_model", best.params.get("d_model_mult", 8) * best.params.get("n_head", 8) * 8)

best_hp = {
    "timestamp": time_stamp,
    "study_name": study_name,
    "best_trial": best.number,
    "best_auc": best.value,
    "n_trials_completed": len([t for t in study.trials if t.state.name == "COMPLETE"]),
    "n_trials_pruned": len([t for t in study.trials if t.state.name == "PRUNED"]),
    "hyperparameters": {
        "D_MODEL": d_model,
        "N_HEAD": best.params["n_head"],
        "N_LAYER": best.params["n_layer"],
        "CONTINUOUS_PROJ_DIM": best.params["continuous_proj_dim"],
        "CLASSIFIER_HIDDEN_DIM": best.params["classifier_hidden_dim"],
        "STATIC_PROJ_DIM": best.params["static_proj_dim"],
        "DROPOUT": best.params["dropout"],
        "LEARNING_RATE": best.params["learning_rate"],
        "WEIGHT_DECAY": best.params["weight_decay"],
        "POS_WEIGHT": best.params["pos_weight"],
        "MASK_PROB": best.params["mask_prob"],
        "MLM_WEIGHT": best.params["mlm_weight"],
        "NUM_EPOCHS": best.params["num_epochs"],
        "TRAIN_BATCH_SIZE": best.params["train_batch_size"],
    },
}

hp_path = f"optuna_best_hp_{time_stamp}.json"
with open(hp_path, "w") as f:
    json.dump(best_hp, f, indent=2)

print(f"Best hyperparameters saved to {hp_path}")
print(f"\nCopy these into model_training_BCE.ipynb config cell:")
print("=" * 60)
for k, v in best_hp["hyperparameters"].items():
    if isinstance(v, float) and v < 0.01:
        print(f"{k} = {v}")
    else:
        print(f"{k} = {v}")
print("=" * 60)